In [1]:
# Import libraries and set up file paths
import pandas as pd
from pathlib import Path

RAW = Path("../dataset/raw")
PROCESSED = Path("../dataset/processed")


In [2]:
#ONE TIME RUN
#make the processed folder
PROCESSED.mkdir(parents=True, exist_ok=True)

In [3]:
# Load the raw train, history, and complaints datasets
train = pd.read_csv(RAW / 'train.csv')
history = pd.read_csv(RAW / 'patient_history.csv')
complaints = pd.read_csv(RAW / 'chief_complaints.csv')


In [4]:
# Remove duplicate columns and merge into one massive dataframe
history = history.drop(columns=[c for c in history.columns if c in train.columns and c != 'patient_id'])
complaints = complaints.drop(columns=[c for c in complaints.columns if c in train.columns and c != 'patient_id'])
complaints = complaints.drop(columns=[c for c in complaints.columns if c in history.columns and c != 'patient_id'])

merged = train.merge(history, on='patient_id', how='left').merge(complaints, on='patient_id', how='left')


In [5]:
#ONE TIME RUN
csv_path = PROCESSED / "schema_preview.csv"

# Check if the file already exists
if not csv_path.exists():
    preview.to_csv(csv_path, index=False)
    print("File created.")
else:
    print("File already exists!")

File already exists!


## Feature Selection & Engineering

We will separate our data into features (`X`) and our target (`y`), which is `triage_acuity`.
We will also drop features that could lead to data leakage or introduce bias.

In [6]:
columns_to_drop = [
    # Data Leakage: These happen AFTER triage is completed
    'disposition',
    'ed_los_hours',
    # Bias / ID fields: We shouldn't predict based on who the patient is
    'patient_id'
]

clean_df = merged.drop(columns=columns_to_drop)
print(f"Dropped {len(columns_to_drop)} columns. Remaining columns: {len(clean_df.columns)}")


Dropped 3 columns. Remaining columns: 63


## Save Processed Datasets (Parquet format)

We save two versions:
1.  **Base Dataset:** Numerical/Categorical features only (no raw NLP text).
2.  **NLP Dataset:** Contains the `chief_complaint_raw` text.

In [7]:
# Base dataset (drop NLP column)
triage_base = clean_df.drop(columns=['chief_complaint_raw'])
triage_base.to_parquet(PROCESSED / 'triage_base.parquet', index=False)
print("Saved triage_base.parquet")

# NLP dataset (includes chief_complaint_raw)
triage_nlp = clean_df.copy()
triage_nlp.to_parquet(PROCESSED / 'triage_nlp.parquet', index=False)
print("Saved triage_nlp.parquet")

Saved triage_base.parquet


Saved triage_nlp.parquet
